In [1]:
import panel as pn
import holoviews as hv
import pandas as pd
from corePlanner import get_targets, get_target_airmass, get_sun_rise_set, parse_ephemeris, determine_moon_phase, event_tonight
import PES_secrets

pn.extension('tabulator')
# Ensure Panel is initialized
pn.extension()
global target_df

In [ ]:
# this cell loads the targets for the night
datestr =pd.Timestamp.now().strftime("%Y-%m-%d")
targets_df = get_targets()  # Fetch the targets DataFrame

In [3]:
# this cell sets up some other text boxes for details of ther night
# get the sunset and sunrise times
sunset, sunrise = get_sun_rise_set(datestr)
# convert the sunset from a timestAMP IN SECONDS TO timezone of utc to local time
sunset = pd.to_datetime(sunset, unit='s').tz_localize('UTC').tz_convert(PES_secrets.obszone)
# convert the sunrise from a timezone of utc to local time
sunrise = pd.to_datetime(sunrise, unit='s').tz_localize('UTC').tz_convert(PES_secrets.obszone)
# create a str pane for the sunset time where there is a title centered and below that the value
sunset_pane = pn.pane.Markdown(f"Sunset\n {sunset.strftime('%H:%M:%S')}", width=150)
# create a str pane for the sunrise time
sunrise_pane = pn.pane.Markdown(f"Sunrise\n {sunrise.strftime('%H:%M:%S')}", width=150)
# create a str pane for the moon phase
moon_phase = determine_moon_phase(datestr)
# create a str pane for the moon phase phase to 0 dp
moon_phase_pane = pn.pane.Markdown(f"Moon Phase\n {moon_phase:.1f}", width=150)

In [ ]:

targets_df = get_targets()  # Fetch the targets DataFrame
#targets_df = targets_df[10:20]
# while testing, set the targets_df to a small subset
#targets_df = targets_df.head(5)
# initialize the ephemeris column null string
targets_df['ephemeris'] = None
targets_df['event'] = None
targets_df['next_event'] = None


In [5]:

# Create a Tabulator widget for interactive row selection
# only show the columns that are needed
target_table_df = targets_df[['star_name', 'ra', 'dec', 'var_type', 'min_mag', 'max_mag', 'period', 'event']]
# convert ra from degrees to hh:mm:ss
target_table_df['ra'] = targets_df['ra'].apply(lambda x: f"{int(x // 15):02}:{int((x % 15) * 4):02}:{int(((x % 15) * 4 % 1) * 60):02}")
# convert dec from degrees to dd:mm:ss
target_table_df['dec'] = targets_df['dec'].apply(lambda x: f"{int(x):02}:{int(abs(x) % 1 * 60):02}:{int((abs(x) % 1 * 60 % 1) * 60):02}")
# create a Tabulator widget for the targets table
targets_table = pn.widgets.Tabulator(
    target_table_df,
    selectable=1,  # Allow single row selection
    width=800,
    height=400
)

# set the title of targets_table
targets_table.title = "Targets for " + pd.Timestamp.now().strftime("%Y-%m-%d")


# Function to handle row selection
def on_row_select(event):
    selected_row = targets_table.selection
    if selected_row:
        selected_data = targets_df.iloc[selected_row[0]]  # Get the selected row data
        print("Selected Row Data:", selected_data)  # Replace with desired action
        # Update the display with selected row data

# Attach the row selection event to the Tabulator widget
targets_table.param.watch(on_row_select, 'selection')

# create a pane to display the selected row data
selected_row_pane = pn.pane.Str("No row selected", width=800)
# Update the pane with selected row data
def update_selected_row_pane(event):
    selected_row = targets_table.selection
    if selected_row:
        selected_data = targets_df.iloc[selected_row[0]]
        selected_row_pane.object = str(selected_data)
    else:
        selected_row_pane.object = "No row selected"
# Attach the update function to the Tabulator widget
targets_table.param.watch(update_selected_row_pane, 'selection')

# here is the function to generate the airmass graph
def generate_airmass_graph(selected_data):
    timezone = PES_secrets.obszone
    times, airmass = get_target_airmass(selected_data)
    # filter out invalid airmass values > 10 and < 0 as None
    airmass = [None if (x > 10 or x < 0) else x for x in airmass]
    
    if not airmass:
        return hv.Text(0.5, 0.5, "No valid data for airmass graph").opts(
            width=800, height=400
        )
    # Convert times to timezone-aware datetime objects
    times = pd.to_datetime(times, unit='s', utc=True).tz_convert(timezone)
    #print(f"Times: {times}")
    #print(f"Airmass: {airmass}")

    # Create a Holoviews plot of airmass vs times
    # so... the scatter object uses tz.naive so the tz must be removed
    airmass_pts = hv.Scatter((times, airmass), label=selected_data['star_name']).opts(
        title="Airmass vs Time",
        xlabel="Time",
        ylabel="Airmass",
        width=800,
        height=400,    
        invert_yaxis=True,
        ylim=(1, 3),
        size=5,
        tools=['hover']
    )   
    airmass_curve = hv.Curve((times, airmass), label=selected_data['star_name']).opts(
         line_width=2
    )
    airmass_graph = airmass_pts * airmass_curve
    # return airmass_graph
 

    # Add vertical lines for ephemeris times
    ephemeris = selected_data.get('ephemeris', None)
    if ephemeris is not None:
        ephemeris_times = [pd.to_datetime(ephem, utc=True).tz_convert(timezone) for ephem in ephemeris]
        for ephem_time in ephemeris_times:
            # convert to tz naive
            ephem_time = ephem_time.tz_localize(None)
            # add a vertical line to the graph
            airmass_graph *= hv.VLine(ephem_time).opts(
                line_color='red',
                line_width=2,
                line_dash='dashed'
            )
    # add a vertical line for the sunset time
    sunset_time = pd.to_datetime(sunset, unit='s', utc=True).tz_convert(timezone)
    # convert to tz naive
    sunset_time = sunset_time.tz_localize(None)
    # add a vertical line to the graph
    airmass_graph *= hv.VLine(sunset_time).opts(
        line_color='orange',
        line_width=2,
        line_dash='dashed'
    )
    # add a vertical line for the sunrise time
    sunrise_time = pd.to_datetime(sunrise, unit='s', utc=True).tz_convert(timezone)
    sunrise_time = sunrise_time.tz_localize(None)
    airmass_graph *= hv.VLine(sunrise_time).opts(
        line_color='orange',
        line_width=2,
        line_dash='dashed'
    )

    return airmass_graph
 

# # Create a pane to display the airmass graph
airmass_pane = pn.pane.HoloViews(
    generate_airmass_graph(targets_df.iloc[0]),  # Initial graph with the first target
    width=800,
    height=400
)

# Function to update the airmass graph based on selected row
def update_airmass_graph(event):
    selected_row = targets_table.selection
    if selected_row:
        selected_data = targets_df.iloc[selected_row[0]]
        # call generate_airmass_graph function to create the graph
        airmass_graph = generate_airmass_graph(selected_data) 
        # Here you would generate the airmass graph based on selected_data
        # For demonstration, we'll just update the pane with a placeholder message
        airmass_pane.object = airmass_graph
# Attach the update function to the Tabulator widget
targets_table.param.watch(update_airmass_graph, 'selection')
# Create a layout for the airmass graph
airmass_layout = pn.Column(
    airmass_pane
)



/tmp/ipykernel_2719652/24718100.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_table_df['ra'] = targets_df['ra'].apply(lambda x: f"{int(x // 15):02}:{int((x % 15) * 4):02}:{int(((x % 15) * 4 % 1) * 60):02}")
/tmp/ipykernel_2719652/24718100.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_table_df['dec'] = targets_df['dec'].apply(lambda x: f"{int(x):02}:{int(abs(x) % 1 * 60):02}:{int((abs(x) % 1 * 60 % 1) * 60):02}")


In [6]:
# now enrich the targets_df with event information
# create a widget to show the progress of the ephemeris parsing
# create a progress bar as a pane in the layout

# set the progress bar to be 0 to len(targets_df)

# create the progress bar
ephem_progress_value = 0
ephem_progress = pn.widgets.Progress(
    value=ephem_progress_value,
    active=True,
    max=len(targets_df),
    width=150,
    bar_color='primary',
    height=20
)
# set the title of the progress bar
ephem_progress.title = "Parsing Ephemeris"

# update the progress bar
def update_progress_bar(value):
    ephem_progress.value = value

# create a button to start the ephemeris parsing
process_ephem_button = pn.widgets.Button(
    name='Process Ephemeris',
    button_type='primary',
    width=150,
    height=40
)

# Function to process ephemeris for each target
def process_ephemeris(event):
    # set the process button to disabled
    process_ephem_button.disabled = True
    # set the progress bar to 0
    ephem_progress.value = 0
    # set the progress bar to active
    ephem_progress.active = True
    # Iterate over each target and parse ephemeris
    for index, row in targets_df.iterrows():
        # print a progress message
        #print(f"Processing target {index + 1} of {len(targets_df)}: {row['star_name']}")
        # check if the ephemeris is empty
        if row['other_info'] is None:
            continue
        # add the ephemeris to the targetdf
        targets_df.at[index, 'ephemeris'] = parse_ephemeris(row['other_info'])
    # update the progress bar
        update_progress_bar(index + 1)
    # run the function event_tonight
    neue_targets_df = event_tonight(targets_df)
    # when the ephemeris is done, set the button to enabled
    process_ephem_button.disabled = False
    # show the required columns in the targets table
    # only show the columns that are needed
    target_table_df = neue_targets_df[['star_name', 'ra', 'dec', 'var_type', 'min_mag', 'max_mag', 'period', 'event']]
    # convert ra from degrees to hh:mm:ss
    target_table_df['ra'] = neue_targets_df['ra'].apply(lambda x: f"{int(x // 15):02}:{int((x % 15) * 4):02}:{int(((x % 15) * 4 % 1) * 60):02}")
    # convert dec from degrees to dd:mm:ss
    target_table_df['dec'] = neue_targets_df['dec'].apply(lambda x: f"{int(x):02}:{int(abs(x) % 1 * 60):02}:{int((abs(x) % 1 * 60 % 1) * 60):02}")
    # update the targets table
    targets_table.value = target_table_df
    # enable the show events tonight button 
    show_events_button.disabled = False
    #print the number of targets
    print(f"Processed {len(targets_df)} targets")

# reset the index to start from 0
#targets_df.reset_index(drop=True, inplace=True)


# set the on_click event of the button to start the ephemeris parsing
process_ephem_button.on_click(process_ephemeris)


Watcher(inst=Button(button_type='primary', height=40, name='Process Ephemeris', sizing_mode='fixed', width=150), cls=<class 'panel.widgets.button.Button'>, fn=<function process_ephemeris at 0x73ffab27cb80>, mode='args', onlychanged=False, parameter_names=('clicks',), what='value', queued=False, precedence=0)

In [7]:
# create a button 'show events tonight'
# initially set to disabled
# it becomes active when the ephemeris is processed
show_events_button = pn.widgets.Button(
    name='Show Events Tonight',
    button_type='primary',
    width=150,
    height=40,
    disabled=True
)

# when show events button is clicked, show the events in the targets table
def show_events_tonight(event):
    # set the show events button to disabled
    show_events_button.disabled = True
    # for each row in the targets_df, check if the event is not None
    for index, row in targets_df.iterrows():
        # check if the event is not None
        if row['event'] is None:
            # drop this row from the targets_df
            targets_df.drop(index, inplace=True)
    # reset the index to start from 0
    targets_df.reset_index(drop=True, inplace=True)
    # show the required columns in the targets table
    # only show the columns that are needed
    target_table_df = targets_df[['star_name', 'ra', 'dec', 'var_type', 'min_mag', 'max_mag', 'period', 'event']]
    # convert ra from degrees to hh:mm:ss
    target_table_df['ra'] = targets_df['ra'].apply(lambda x: f"{int(x // 15):02}:{int((x % 15) * 4):02}:{int(((x % 15) * 4 % 1) * 60):02}")
    # convert dec from degrees to dd:mm:ss
    target_table_df['dec'] = targets_df['dec'].apply(lambda x: f"{int(x):02}:{int(abs(x) % 1 * 60):02}:{int((abs(x) % 1 * 60 % 1) * 60):02}")
    # update the targets table
    targets_table.value = target_table_df
# set the on_click event of the button to show events
show_events_button.on_click(show_events_tonight)

Watcher(inst=Button(button_type='primary', disabled=True, height=40, name='Show Events Tonight', sizing_mode='fixed', width=150), cls=<class 'panel.widgets.button.Button'>, fn=<function show_events_tonight at 0x73ffab27d000>, mode='args', onlychanged=False, parameter_names=('clicks',), what='value', queued=False, precedence=0)

In [8]:
# create a reset_targets_btton to reset the targets_df
reset_targets_button = pn.widgets.Button(
    name='Reset Targets',
    button_type='primary',
    width=150,
    height=40
)
# when reset targets button is clicked, reset the targets_df
def reset_targets(event):
    # reset the targets_df to the original targets_df
    global targets_df
    # reset the targets_df to the original targets_df
    targets_df = get_targets()
    # initialize the ephemeris column null string
    targets_df['ephemeris'] = None
    targets_df['event'] = None
    # show the required columns in the targets table
    # only show the columns that are needed
    target_table_df = targets_df[['star_name', 'ra', 'dec', 'var_type', 'min_mag', 'max_mag', 'period', 'event']]
    # convert ra from degrees to hh:mm:ss
    target_table_df['ra'] = targets_df['ra'].apply(lambda x: f"{int(x // 15):02}:{int((x % 15) * 4):02}:{int(((x % 15) * 4 % 1) * 60):02}")
    # convert dec from degrees to dd:mm:ss
    target_table_df['dec'] = targets_df['dec'].apply(lambda x: f"{int(x):02}:{int(abs(x) % 1 * 60):02}:{int((abs(x) % 1 * 60 % 1) * 60):02}")
    # update the targets table
    targets_table.value = target_table_df
    # reset the progress bar
    ephem_progress.value = 0
    # reset the process button to enabled
    process_ephem_button.disabled = False
    # reset the show events button to disabled
    show_events_button.disabled = True
    # reset the selected row pane to no row selected
    selected_row_pane.object = "No row selected"
# set the on_click event of the button to reset targets
reset_targets_button.on_click(reset_targets)

Watcher(inst=Button(button_type='primary', height=40, name='Reset Targets', sizing_mode='fixed', width=150), cls=<class 'panel.widgets.button.Button'>, fn=<function reset_targets at 0x73ffab27d120>, mode='args', onlychanged=False, parameter_names=('clicks',), what='value', queued=False, precedence=0)

In [ ]:

# create a Panel layout
layout = pn.Column(
    pn.pane.Markdown("## AAVSO Targets for " + pd.Timestamp.now().strftime("%Y-%m-%d")),
    pn.Row(
        pn.Column(
            process_ephem_button,
            ephem_progress,
            show_events_button,
            reset_targets_button,
            sunset_pane,
            sunrise_pane,
            moon_phase_pane
    ),
        
        targets_table,
        airmass_layout
    ),
    pn.Row(
        pn.pane.Markdown("### Selected Row Data"),
        selected_row_pane
    )
)
# Display the layout in a Jupyter notebook
layout.servable()
# Alternatively, if you want to run this as a standalone script,
# you can use the following line to serve the panel:
pn.serve(layout, show=True)


Launching server at http://localhost:34477


/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonz

Selected Row Data: star_name                                                         SU Men
ra                                                              61.66346
dec                                                            -77.13267
constellation                                                        Men
var_type                                                              EA
min_mag                                                             14.5
min_mag_band                                                           p
max_mag                                                             13.0
max_mag_band                                                           p
period                                                               NaN
obs_cadence                                                          3.0
obs_mode                                                             All
obs_section                                        [Eclipsing Variables]
filter                          

/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonz

Selected Row Data: star_name                                                         VY Ret
ra                                                              51.81983
dec                                                            -61.25914
constellation                                                        Ret
var_type                                                              EA
min_mag                                                             8.47
min_mag_band                                                           V
max_mag                                                             7.89
max_mag_band                                                           V
period                                                          14.21605
obs_cadence                                                     1.421605
obs_mode                                                             All
obs_section                                        [Eclipsing Variables]
filter                          

...

/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonz

Selected Row Data: star_name                                                GSC 00814-00323
ra                                                             133.04392
dec                                                             12.29831
constellation                                                        Cnc
var_type                                                              EA
min_mag                                                             10.0
min_mag_band                                                           V
max_mag                                                             9.67
max_mag_band                                                           V
period                                                            65.864
obs_cadence                                                       6.5864
obs_mode                                                             All
obs_section                                        [Eclipsing Variables]
filter                          

/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonz

Selected Row Data: star_name                                                         BV Ant
ra                                                              165.3562
dec                                                             -37.1718
constellation                                                        Ant
var_type                                                              EA
min_mag                                                            12.24
min_mag_band                                                           V
max_mag                                                            11.52
max_mag_band                                                           V
period                                                           3.59426
obs_cadence                                                     0.359426
obs_mode                                                             All
obs_section                                        [Eclipsing Variables]
filter                          

/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonzero nanoseconds in conversion.
  return (cmin.to_pydatetime().replace(tzinfo=None),
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:216: UserWarning: Discarding nonzero nanoseconds in conversion.
  cmax.to_pydatetime().replace(tzinfo=None))
/home/adminuser/Documents/AstroPlanner/.env/lib/python3.10/site-packages/holoviews/core/data/pandas.py:215: UserWarning: Discarding nonz